# 04 - qf_mtp_eval 项目架构解析

## 学习目标
- 理解 qf_mtp_eval 的整体架构和模块组织
- 掌握 Deploy → Eval → Cleanup 三阶段流程
- 理解数据库模型和 Benchmark 注册机制
- 实践：手动解析 deploy script

## 1. 项目整体架构

```
qf_mtp_eval/
├── fastmtp_eval/              # 核心 Python 包
│   ├── cli.py                 # CLI 入口 (Click)
│   ├── api.py                 # FastAPI Web 服务
│   ├── config.py              # 配置管理
│   ├── web_service.py         # Web 评测服务编排
│   ├── core/
│   │   ├── task_manager.py    # 任务 CRUD
│   │   └── async_executor.py  # 异步执行器
│   ├── engines/
│   │   ├── deploy_engine.py   # 部署引擎 (启动 SGLang)
│   │   ├── eval_engine.py     # 评测引擎 (运行 benchmark)
│   │   └── cleanup_engine.py  # 清理引擎 (杀进程、释放端口)
│   ├── parsers/
│   │   ├── sglang_parser.py   # SGLang 脚本解析器
│   │   └── bench_parser.py    # Benchmark 脚本解析器
│   ├── connectors/            # 远程执行连接器
│   │   ├── local.py           # 本地执行
│   │   ├── ssh.py             # SSH 远程执行
│   │   └── kubectl_exec.py    # K8s Pod 内执行
│   └── db/
│       ├── models.py          # SQLAlchemy 数据模型
│       └── session.py         # 数据库会话
│
├── evaluation_dev/            # 评测开发目录
│   ├── framework/
│   │   └── runner.py          # 统一 Runner (核心!)
│   ├── common/
│   │   └── bench_sglang_chat.py  # 通用 Chat Benchmark
│   ├── aiak_bench/            # AIAK 评测集
│   ├── math_500/              # Math-500
│   ├── c_eval/                # C-Eval
│   ├── spec_bench/            # SpecBench
│   └── experiments/examples/  # 实验配置示例
│
└── deploy_scripts/            # 模型部署脚本
    ├── ds_align/              # DeepSeek 系列
    └── glm5/                  # GLM-5 系列
```

## 2. 核心流程：Deploy → Eval → Cleanup

```
┌─────────────────────────────────────────────────────────────────┐
│                    Task Lifecycle                                 │
├─────────────┬──────────────────┬────────────────────────────────┤
│  Phase 1    │   Phase 2        │   Phase 3                      │
│  DEPLOY     │   EVALUATE       │   CLEANUP                      │
├─────────────┼──────────────────┼────────────────────────────────┤
│             │                  │                                │
│ 1. 解析     │ 1. 加载 bench   │ 1. kill server process         │
│    deploy   │    配置文件      │    (SIGTERM → SIGKILL)         │
│    script   │                  │                                │
│             │ 2. 构建 bench   │ 2. 释放端口                    │
│ 2. 找可用   │    命令行参数    │                                │
│    端口     │                  │ 3. 收集最终结果                │
│             │ 3. 执行 bench   │                                │
│ 3. 启动     │    脚本          │                                │
│    SGLang   │                  │                                │
│    server   │ 4. 解析输出     │                                │
│             │    result.jsonl  │                                │
│ 4. 等待     │                  │                                │
│    健康检查 │ 5. 写入数据库    │                                │
│    通过     │                  │                                │
│             │                  │                                │
└─────────────┴──────────────────┴────────────────────────────────┘
```

### 两条执行路径

1. **Web/CLI 路径**: `AsyncExecutor` → DeployEngine → EvalEngine → CleanupEngine
2. **Runner 路径**: `framework/runner.py` — 独立的统一 runner，读取 experiment.json 驱动全流程

## 3. 核心模块解析

### 3.1 DeployEngine — 部署引擎

**文件**: `fastmtp_eval/engines/deploy_engine.py`

职责:
- 解析 deploy shell script (通过 `SGLangScriptParser`)
- 自动找可用端口 (`find_available_port`)
- 启动 SGLang server 子进程
- 等待服务健康检查通过 (`wait_for_port`)
- 提供 stop/kill 接口

In [ ]:
# 查看 DeployEngine 的核心逻辑（源码阅读）
# 路径: /mnt/cfs_bj_mt/workspace/limengjie03/tool_chain/qf_mtp_eval/fastmtp_eval/engines/deploy_engine.py

# 简化版本的 deploy 逻辑
import subprocess
import os
import time
import signal
import socket


def find_available_port(start=30000, end=31000):
    """Find an available port in the given range."""
    for port in range(start, end):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(("", port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"No available port in range [{start}, {end})")


def wait_for_port(port, host="127.0.0.1", timeout=300, interval=5):
    """Wait until a port is accepting connections."""
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.connect((host, port))
                return True
            except (ConnectionRefusedError, OSError):
                time.sleep(interval)
    return False


print("DeployEngine utilities loaded.")
port = find_available_port()
print(f"Found available port: {port}")

### 3.2 SGLangScriptParser — 脚本解析器

**文件**: `fastmtp_eval/parsers/sglang_parser.py`

职责:
- 解析 bash shell script 中的 `sglang.launch_server` 命令
- 提取所有参数到结构化的 `SGLangDeployConfig` 数据类
- 处理 backslash continuation、环境变量等 shell 语法

In [ ]:
import re
import shlex
from dataclasses import dataclass, field
from typing import Any, Optional


@dataclass
class SGLangDeployConfig:
    """Parsed SGLang deployment configuration.
    
    This is a simplified version of the actual class in qf_mtp_eval.
    See: fastmtp_eval/parsers/sglang_parser.py
    """
    script_path: str = ""
    model_path: Optional[str] = None
    host: Optional[str] = None
    port: Optional[int] = None
    tp_size: Optional[int] = None
    dp_size: Optional[int] = None
    enable_dp_attention: bool = False
    mem_fraction_static: Optional[float] = None
    max_running_requests: Optional[int] = None
    speculative_algorithm: Optional[str] = None
    speculative_num_steps: Optional[int] = None
    speculative_eagle_topk: Optional[int] = None
    speculative_num_draft_tokens: Optional[int] = None
    enable_multi_layer_eagle: bool = False
    reasoning_parser: Optional[str] = None
    env_vars: dict = field(default_factory=dict)
    extra_args: list = field(default_factory=list)


def parse_sglang_script(content: str) -> SGLangDeployConfig:
    """Parse a SGLang deploy script content.
    
    Handles:
    - Backslash line continuations
    - Environment variable exports
    - Command-line arguments in various formats (--arg value, --arg=value)
    """
    config = SGLangDeployConfig()
    
    # Handle line continuations
    content = re.sub(r"\\\s*\n", " ", content)
    
    # Extract env vars
    for match in re.finditer(r"export\s+([A-Z_][A-Z0-9_]*)\s*=\s*(.+)", content):
        config.env_vars[match.group(1)] = match.group(2).strip("'\"")
    
    # Find sglang command
    for line in content.split("\n"):
        if "sglang.launch_server" in line:
            # Parse this command line
            parts = shlex.split(line)
            i = 0
            while i < len(parts):
                arg = parts[i]
                if arg == "--model-path" and i+1 < len(parts):
                    config.model_path = parts[i+1]; i += 1
                elif arg == "--host" and i+1 < len(parts):
                    config.host = parts[i+1]; i += 1
                elif arg == "--port" and i+1 < len(parts):
                    config.port = int(parts[i+1]); i += 1
                elif arg == "--tp-size" and i+1 < len(parts):
                    config.tp_size = int(parts[i+1]); i += 1
                elif arg == "--dp-size" and i+1 < len(parts):
                    config.dp_size = int(parts[i+1]); i += 1
                elif arg == "--enable-dp-attention":
                    config.enable_dp_attention = True
                elif arg == "--mem-fraction-static" and i+1 < len(parts):
                    config.mem_fraction_static = float(parts[i+1]); i += 1
                elif arg == "--speculative-algorithm" and i+1 < len(parts):
                    config.speculative_algorithm = parts[i+1]; i += 1
                elif arg.startswith("--speculative-num-steps"):
                    val = arg.split("=")[1] if "=" in arg else parts[i+1]
                    config.speculative_num_steps = int(val)
                    if "=" not in arg: i += 1
                elif arg.startswith("--speculative-eagle-topk"):
                    val = arg.split("=")[1] if "=" in arg else parts[i+1]
                    config.speculative_eagle_topk = int(val)
                    if "=" not in arg: i += 1
                elif arg.startswith("--speculative-num-draft-tokens"):
                    val = arg.split("=")[1] if "=" in arg else parts[i+1]
                    config.speculative_num_draft_tokens = int(val)
                    if "=" not in arg: i += 1
                elif arg == "--reasoning-parser" and i+1 < len(parts):
                    config.reasoning_parser = parts[i+1]; i += 1
                i += 1
    
    return config


print("SGLangScriptParser loaded.")

In [ ]:
# 实战：解析一个真实的 deploy script
# 这是来自 qf_mtp_eval 的 deploy_scripts/ds_align/start_ds_0305_eagle.sh

sample_script = """
python3 -m sglang.launch_server \\
        --model-path /mnt/cfs_bj_mt/workspace/liulinyun/work2026/01/0119_mtp_train/outputs/v0.0.11.260324-dsv32-think/checkpoints/iter_0001900_hf \\
        --dp-size 8 \\
        --enable-dp-attention \\
        --tp-size 8 \\
        --trust-remote-code \\
        --mem-fraction-static 0.75 \\
        --max-running-requests 256 \\
        --cuda-graph-max-bs 32 \\
        --chunked-prefill-size 16384 \\
        --speculative-algorithm EAGLE \\
        --speculative-num-steps=3 \\
        --speculative-eagle-topk=1 \\
        --reasoning-parser deepseek-v3 \\
        --speculative-num-draft-tokens=4 \\
        --max-total-tokens 500000 \\
        --host 0.0.0.0 --port 30000
"""

config = parse_sglang_script(sample_script)

print("=== Parsed Deploy Config ===")
print(f"Model path:      {config.model_path}")
print(f"Host:Port:       {config.host}:{config.port}")
print(f"TP size:         {config.tp_size}")
print(f"DP size:         {config.dp_size}")
print(f"DP Attention:    {config.enable_dp_attention}")
print(f"Mem fraction:    {config.mem_fraction_static}")
print(f"\n--- Speculative Decoding ---")
print(f"Algorithm:       {config.speculative_algorithm}")
print(f"Num steps:       {config.speculative_num_steps}")
print(f"EAGLE topk:      {config.speculative_eagle_topk}")
print(f"Num draft tokens:{config.speculative_num_draft_tokens}")
print(f"Reasoning parser:{config.reasoning_parser}")

### 3.3 framework/runner.py — 统一 Runner

**文件**: `evaluation_dev/framework/runner.py`

这是最核心的组件，负责：
1. 解析 experiment.json 配置
2. 启动 service (调用 deploy script)
3. 等待 service 就绪 (health check)
4. 逐个运行 benchmark tasks
5. 收集结果
6. 停止 service

### experiment.json 结构

```json
{
  "experiment": {
    "name": "single-model-eval",
    "continue_on_error": false
  },
  "service": {
    "command_file": "/path/to/start_eagle.sh",
    "ready_port": 30000,
    "ready_url": "http://127.0.0.1:30000/health",
    "startup_timeout_sec": 300
  },
  "tasks": [
    {
      "id": "aiak_single",
      "bench": "aiak_bench",
      "args": {"port": 30000, "thinking": false}
    },
    {
      "id": "math_thinking",
      "bench": "math_500",
      "args": {"port": 30000, "thinking": true}
    }
  ]
}
```

In [ ]:
import json

# 理解 experiment.json 的结构
experiment_example = {
    "experiment": {
        "name": "my-first-eval",
        "continue_on_error": False,  # 某个 task 失败是否继续
    },
    "service": {
        "command_file": "/path/to/start_eagle.sh",  # 部署脚本路径
        "ready_port": 30000,                          # 服务监听端口
        "ready_url": "http://127.0.0.1:30000/health", # 健康检查 URL
        "startup_timeout_sec": 300,                    # 启动超时
    },
    "tasks": [
        {
            "id": "aiak_basic",      # 任务唯一 ID
            "bench": "aiak_bench",   # 对应 bench.toml 的 ID
            "args": {                 # 覆盖 bench 默认参数
                "port": 30000,
                "thinking": False,
                "num-questions": 100,
            },
        },
    ],
}

print(json.dumps(experiment_example, indent=2))

### 3.4 bench.toml — Benchmark 注册机制

每个 benchmark 目录下都有一个 `bench.toml` 文件，声明该 benchmark 的元信息。

```toml
[bench]
id = "aiak_bench"
title = "AIAK Bench"
description = "Conversation benchmark with single-turn and multi-turn filtering."

[runner]
script = "/path/to/bench_sglang_eagle.py"

[runner.args]
question-file = "/path/to/eval_10w_2k.jsonl"
parallel = 16
host = "http://127.0.0.1"
num-questions = 2000
temperature = 0
max-gen-length = 8192
mt = "all"

[runner.outputs]
answer-file = "answers.jsonl"
result-file = "result.jsonl"
```

### Runner 如何使用 bench.toml

1. 根据 task 的 `bench` 字段找到对应目录的 `bench.toml`
2. 读取 `runner.script` 作为执行入口
3. 合并 `runner.args` (默认参数) 和 task 的 `args` (覆盖参数)
4. 输出文件使用 `runner.outputs` 定义的名称

In [ ]:
# 模拟 Runner 解析 bench.toml 的逻辑
try:
    import tomllib  # Python 3.11+
except ImportError:
    import tomli as tomllib  # fallback


def load_bench_manifest(bench_toml_content: str) -> dict:
    """Parse a bench.toml manifest."""
    return tomllib.loads(bench_toml_content)


def resolve_bench_command(manifest: dict, task_args: dict, port: int) -> list:
    """Build the benchmark execution command from manifest + task args.
    
    This mimics what framework/runner.py does internally.
    """
    script = manifest["runner"]["script"]
    default_args = manifest["runner"].get("args", {})
    outputs = manifest["runner"].get("outputs", {})
    
    # Merge: task_args override default_args
    merged = {**default_args, **task_args}
    
    # Build command
    cmd = ["python3", script]
    for key, value in merged.items():
        if isinstance(value, bool):
            if value:
                cmd.append(f"--{key}")
        else:
            cmd.extend([f"--{key}", str(value)])
    
    # Add output files
    for key, value in outputs.items():
        cmd.extend([f"--{key}", value])
    
    return cmd


# 示例
bench_toml = """
[bench]
id = "aiak_bench"
title = "AIAK Bench"

[runner]
script = "/path/to/bench_sglang_eagle.py"

[runner.args]
question-file = "/path/to/eval_10w_2k.jsonl"
parallel = 16
host = "http://127.0.0.1"
num-questions = 2000
temperature = 0
max-gen-length = 8192

[runner.outputs]
answer-file = "answers.jsonl"
result-file = "result.jsonl"
"""

manifest = load_bench_manifest(bench_toml)
task_args = {"port": 30000, "num-questions": 100, "thinking": True}

cmd = resolve_bench_command(manifest, task_args, port=30000)
print("Generated command:")
print(" \\\n    ".join(cmd))

## 4. 数据库模型

**文件**: `fastmtp_eval/db/models.py`

```python
# 核心数据模型（SQLAlchemy ORM）

class Task:
    id: int
    name: str
    status: str             # pending/running/completed/failed
    deploy_config_id: int   # 关联的部署配置
    benchmark_id: int       # 关联的 benchmark
    created_at: datetime
    
class DeployConfig:
    id: int
    name: str
    model_path: str
    script_path: str
    host: str
    port: int
    sglang_params: dict     # JSON 字段存储所有 SGLang 参数
    
class Benchmark:
    id: int
    name: str
    script_path: str
    question_file: str
    params: dict
    
class TaskResult:
    id: int
    task_id: int
    latency: float
    throughput: float
    accept_length: float
    num_requests: int
    result_data: dict       # 完整结果 JSON
```

## 5. Connector 机制

qf_mtp_eval 支持在不同环境中执行命令：

| Connector | 用途 |
|-----------|------|
| `LocalConnector` | 本地 subprocess 执行 |
| `SshConnector` | 通过 SSH 在远程机器执行 |
| `KubectlExecConnector` | 通过 kubectl exec 在 K8s Pod 中执行 |

所有 Connector 实现相同的接口 (`BaseConnector`):

```python
class BaseConnector(ABC):
    def run(self, command: str, timeout: int = None) -> tuple[int, str, str]
    def run_background(self, command: str) -> str  # returns job_id
    def exists(self, path: str) -> bool
    def makedirs(self, path: str) -> None
    def read_file(self, path: str) -> str
    def write_file(self, path: str, content: str) -> None
```

## 6. 可用的 Benchmark 列表

| Benchmark ID | 说明 | 数据类型 |
|-------------|------|----------|
| `aiak_bench` | AIAK 对话评测 (单轮+多轮) | 通用对话 |
| `custom_bench` | 自定义 2k 样本 | 混合 |
| `qianfan_sft_bench` | 千帆 SFT 数据 | SFT 指令 |
| `c_eval` | C-Eval 中文评测 | 选择题 |
| `livecodebench_v6` | 代码生成评测 | 代码 |
| `math_500` | MATH-500 数学 | 数学推理 |
| `mt_bench` | MT-Bench 多轮 | 多轮对话 |
| `spec_bench` | SpecBench 摘要 | 长文本 |

每个 benchmark 都有两个变体：
- `*_eagle` — 使用 SGLang Python SDK (`sgl.function`)
- `*_chat` — 使用 HTTP API (`/v1/chat/completions`)

In [ ]:
# 总结：qf_mtp_eval 的完整数据流

data_flow = """
=== qf_mtp_eval Data Flow ===

1. INPUT
   ├── experiment.json (实验配置)
   ├── deploy_script.sh (SGLang 启动脚本)
   └── bench.toml + question.jsonl (评测配置和数据)

2. DEPLOY PHASE
   ├── parse deploy script → SGLangDeployConfig
   ├── find available port
   ├── launch subprocess: python3 -m sglang.launch_server ...
   └── wait for health check: GET /health → 200 OK

3. EVAL PHASE
   ├── resolve bench.toml → script path + args
   ├── merge task args with bench defaults
   ├── execute: python3 bench_sglang_eagle.py ...
   │   ├── load_questions(jsonl)
   │   ├── run_batch(arguments, temperature=0, max_new_tokens=8192)
   │   ├── collect meta_info per sample
   │   ├── compute accept_length = completion_tokens / spec_verify_ct
   │   └── write answers.jsonl + result.jsonl
   └── parse result.jsonl → TaskResult

4. CLEANUP PHASE
   ├── SIGTERM → wait → SIGKILL (if needed)
   └── release port

5. OUTPUT
   ├── answers.jsonl (每条问答记录 + meta)
   ├── result.jsonl (汇总指标: throughput, latency, accept_length)
   └── TaskResult in SQLite (持久化)
"""
print(data_flow)

## 本节小结

| 知识点 | 掌握内容 |
|--------|----------|
| 项目架构 | fastmtp_eval (核心包) + evaluation_dev (评测) + deploy_scripts |
| 三阶段流程 | Deploy → Eval → Cleanup, 由 AsyncExecutor 或 runner.py 驱动 |
| DeployEngine | 解析 script → 启动进程 → 等待健康检查 |
| SGLangScriptParser | 从 shell script 提取结构化配置 |
| bench.toml | Benchmark 注册和参数声明 |
| experiment.json | 实验全局配置: service + tasks |

---
**下一节**: 05_benchmark_scripts — Benchmark 脚本深度剖析与定制